# M1 — User Segmentation (v2)

## Motivation and approach

The aim is to identify groups of users with homogeneous behaviour towards the email marketing campaigns, in order to personalise strategies in modules M2 (propensity), M3 (recommender) and M4 (forecast).

The segmentation is behavioural: it is based on each user's history of interactions (opens and clicks), not on their demographic attributes. The demographic attributes are used afterwards, in the profiling phase, to characterise the segments.

### Problems identified in v1 and solutions applied

| Problem v1 | Solution v2 |
|---|---|
| 78% of users with 1 event → dominant cluster (45%) | Product category features (`product_new`): 25 categories distinguish users in the same sector |
| Sectors mixed clicks and opens | Engagement features by sector (click_rate within each sector) |
| Skewed counts (n_eventos max=42, median=1) | log1p transformation before scaling |
| 16 features with noise in Euclidean space | PCA to 90% of variance before clustering |
| HDBSCAN with 350 micro-clusters or 60% outliers | Automatic search of `min_cluster_size` for ≤ 20% outliers |
| Silhouette penalised by degenerate KMeans k=2 | K_RANGE from 4; penalty if max_cluster > 60% |

## Libraries

In [ ]:
import warnings
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
import umap

warnings.filterwarnings('ignore')
np.random.seed(42)

# Add src/ to the path and reuse the shared helper (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

ROOT_PATH      = find_project_root()
DATA_PATH      = ROOT_PATH / "data"
PROCESSED_PATH = DATA_PATH / "processed"

print(f"ROOT_PATH: {ROOT_PATH}")

## Data Loading

In [ ]:
df_users    = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str}, low_memory=False)
df_events   = pd.read_csv(PROCESSED_PATH / "events.csv")
df_products = pd.read_csv(PROCESSED_PATH / "products.csv")

df_events["timestamp"] = pd.to_datetime(df_events["timestamp"])

print(f"Usuarios:  {df_users.shape}")
print(f"Eventos:   {df_events.shape}")
print(f"Productos: {df_products.shape}")

# Enrich events with sector and product_new
df_ev = df_events.merge(
    df_products[["id_product", "sector", "product_new"]],
    on="id_product",
    how="left"
)

# Only users that are also in users.csv (geographically enriched)
df_ev = df_ev[df_ev["id_user"].isin(df_users["id_user"])].copy()

print(f"\nEventos tras filtro usuarios: {len(df_ev):,}")
print(f"Usuarios con eventos: {df_ev['id_user'].nunique():,}")

## Feature Engineering

Three blocks of features are built:

1. General activity — volume and recency of the user's interactions.
2. Sector preference — distribution of events by sector and click_rate within each sector.
3. Product category preference — type of product the user interacts with.

The product category block makes it possible to distinguish users with a single event in the same sector according to the specific category (`life_insurance`, `car_insurance`, `health_insurance`, etc.).

In [ ]:
# ── Block 1: Activity metrics ─────────────────────────────────────────────
FECHA_REF = df_ev["timestamp"].max()

# Mark each event as click or open (1/0) so they can be summed without lambdas
df_ev["es_click"] = (df_ev["event_type"] == "click").astype(int)
df_ev["es_open"]  = (df_ev["event_type"] == "open").astype(int)

agg = df_ev.groupby("id_user").agg(
    n_eventos    = ("id_event",   "count"),
    n_clicks     = ("es_click",   "sum"),
    n_opens      = ("es_open",    "sum"),
    n_productos  = ("id_product", "nunique"),
    n_sectores   = ("sector",     "nunique"),
    ultima_fecha = ("timestamp",  "max"),
).reset_index()

# Recency = days from the user's last event to the reference date
agg["recencia_dias"] = (FECHA_REF - agg["ultima_fecha"]).dt.days
agg = agg.drop(columns="ultima_fecha")

agg["click_rate"] = agg["n_clicks"] / agg["n_eventos"]

print("Bloque 1 — actividad:")
display(agg.describe().round(2))

In [ ]:
# ── Block 2: Sector preference (freq) + engagement by sector ──────────────
# The relative frequency by sector is separated from the click_rate by sector,
# to distinguish users who only open from users who click.

SECTORES = [
    "seguros", "energía", "teleco", "motor",
    "servicios", "hogar", "servicios legales",
    "salud y belleza", "finanzas"
]

def col_name(s):
    return s.strip().lower().replace(" ", "_").replace("í", "i").replace("é", "e")

df_sec = df_ev[df_ev["sector"].isin(SECTORES)].copy()

# Relative frequency of events by sector
sec_events = (
    df_sec.groupby(["id_user", "sector"]).size()
    .unstack(fill_value=0)
)
sec_freq = sec_events.div(sec_events.sum(axis=1), axis=0)
sec_freq.columns = ["sec_" + col_name(c) for c in sec_freq.columns]

# Click_rate by sector (clicks in sector / events in sector)
sec_clicks = (
    df_sec[df_sec["event_type"] == "click"]
    .groupby(["id_user", "sector"]).size()
    .unstack(fill_value=0)
)
sec_cr = sec_clicks.div(sec_events.replace(0, np.nan)).fillna(0)
sec_cr.columns = ["cr_" + col_name(c) for c in sec_cr.columns]

df_sector_feat = sec_freq.join(sec_cr, how="outer").fillna(0).reset_index()

print(f"Bloque 2 — features sectoriales: {df_sector_feat.shape[1]-1} columnas")
display(df_sector_feat.head(3))

In [ ]:
# ── Block 3: Product category preference ──────────────────────────────────
# Top-20 categories by event volume; the rest are grouped into 'other'.

top_prods = (
    df_ev["product_new"].value_counts(normalize=True)
    .head(20).index.tolist()
)
print(f"Top-20 categorías de producto ({len(top_prods)} usadas):")
print(top_prods)

df_ev["prod_bucket"] = df_ev["product_new"].where(
    df_ev["product_new"].isin(top_prods), other="other"
)

prod_counts = (
    df_ev.groupby(["id_user", "prod_bucket"]).size()
    .unstack(fill_value=0)
)
prod_freq = prod_counts.div(prod_counts.sum(axis=1), axis=0)
prod_freq.columns = ["prod_" + c for c in prod_freq.columns]
df_prod_feat = prod_freq.reset_index()

print(f"\nBloque 3 — features de producto: {df_prod_feat.shape[1]-1} columnas")

In [ ]:
# ── Final feature matrix ───────────────────────────────────────────────────
df_feat = (
    agg
    .merge(df_sector_feat, on="id_user", how="left")
    .merge(df_prod_feat,   on="id_user", how="left")
    .fillna(0)
)

# Only users with geographic data
df_feat = df_feat[df_feat["id_user"].isin(df_users["id_user"])].reset_index(drop=True).copy()

ACT_COLS  = ["n_eventos", "n_clicks", "n_opens", "n_productos", "n_sectores", "recencia_dias", "click_rate"]
SEC_COLS  = [c for c in df_feat.columns if c.startswith("sec_") or c.startswith("cr_")]
PROD_COLS = [c for c in df_feat.columns if c.startswith("prod_")]
ALL_FEAT  = ACT_COLS + SEC_COLS + PROD_COLS

print(f"Usuarios: {len(df_feat):,}")
print(f"Features totales: {len(ALL_FEAT)}  (actividad={len(ACT_COLS)}, sector={len(SEC_COLS)}, producto={len(PROD_COLS)})")

## Preprocessing

### Transformations prior to scaling

The counts (`n_eventos`, `n_clicks`, `n_opens`, `n_productos`) have heavily right-skewed distributions (median=1, maximum=42), which would cause StandardScaler to give excessive weight to outliers in the Euclidean distance. `log1p` is applied before scaling to compress the scale monotonically. The remaining variables (`click_rate`, `recencia_dias`, frequencies) are already bounded and require no transformation.

In [ ]:
# ── log1p on counts ────────────────────────────────────────────────────────
LOG_COLS = ["n_eventos", "n_clicks", "n_opens", "n_productos"]

X_raw = df_feat[ALL_FEAT].values.copy()
log_idx = [ALL_FEAT.index(c) for c in LOG_COLS]
X_raw[:, log_idx] = np.log1p(X_raw[:, log_idx])

# ── StandardScaler ────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"X_scaled shape: {X_scaled.shape}")

# Check that log1p reduced the skew
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(df_feat["n_eventos"], bins=50, color="steelblue")
axes[0].set_title("n_eventos original (skew alto)")
axes[1].hist(np.log1p(df_feat["n_eventos"]), bins=50, color="darkorange")
axes[1].set_title("log1p(n_eventos) — distribución comprimida")
plt.tight_layout()
plt.show()

### Dimensionality reduction with PCA

With ~50 features and strong correlation between them (frequency and click_rate of the same sector, `prod_seguros` ↔ `sec_seguros`), PCA is applied retaining the components that explain ≥90% of the variance. This reduces noise and speeds up the clustering.

In [ ]:
# ── PCA ───────────────────────────────────────────────────────────────────
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)
var_acum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(var_acum) + 1), var_acum, marker="o", markersize=4, linewidth=2)
for thr in [0.80, 0.90, 0.95]:
    n_c = int(np.argmax(var_acum >= thr)) + 1
    ax.axhline(thr, linestyle="--", alpha=0.6, label=f"{int(thr*100)}% → {n_c} comp.")
ax.set_xlabel("Componentes principales")
ax.set_ylabel("Varianza explicada acumulada")
ax.set_title("PCA — Varianza explicada acumulada")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

N_PCA = int(np.argmax(var_acum >= 0.90)) + 1
print(f"Componentes seleccionados (≥90%): {N_PCA}")

pca = PCA(n_components=N_PCA, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"X_pca shape: {X_pca.shape}")

In [ ]:
# ── UMAP 2D for visualisation ──────────────────────────────────────────────
# Note: UMAP is applied on X_pca (not X_scaled) to speed it up and improve quality
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
X_umap = reducer.fit_transform(X_pca)
print(f"UMAP shape: {X_umap.shape}")

## Selecting k — KMeans

k ∈ [4, 12] is explored; k < 4 is discarded because in v1 k=2 produced a degenerate solution (one cluster with 95% of users). The final selection combines three criteria:

- Silhouette (higher means better inter-cluster separation).
- Davies-Bouldin (lower means better compactness).
- Cluster balance: solutions in which the largest cluster exceeds 60% of the users are penalised.

In [ ]:
K_RANGE = range(4, 13)
km_results = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels, sample_size=5000, random_state=42)
    dbi = davies_bouldin_score(X_pca, labels)
    max_pct = pd.Series(labels).value_counts(normalize=True).max()
    km_results.append({"k": k, "inercia": km.inertia_, "sil": sil, "dbi": dbi, "max_pct": max_pct})
    flag = " ⚠ cluster dominante" if max_pct > 0.60 else ""
    print(f"k={k:2d}  sil={sil:.4f}  DBI={dbi:.3f}  max_cluster={max_pct:.1%}{flag}")

df_km_res = pd.DataFrame(km_results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(df_km_res["k"], df_km_res["inercia"], marker="o")
axes[0].set_title("Método del Codo")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inercia")
axes[0].grid(alpha=0.3)

axes[1].plot(df_km_res["k"], df_km_res["sil"], marker="o", color="green")
axes[1].set_title("Silueta (↑ mejor)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].grid(alpha=0.3)

axes[2].plot(df_km_res["k"], df_km_res["dbi"], marker="o", color="red")
axes[2].set_title("Davies-Bouldin (↓ mejor)")
axes[2].set_xlabel("k")
axes[2].set_ylabel("DBI")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Selecting k: maximum silhouette among the k without a dominant cluster (>60%)
df_valid = df_km_res[df_km_res["max_pct"] <= 0.60]
if df_valid.empty:
    print("Advertencia: todos los k tienen cluster dominante. Usando máxima silueta global.")
    df_valid = df_km_res

K_OPT = int(df_valid.loc[df_valid["sil"].idxmax(), "k"])
print(f"k óptimo (silueta + balance): {K_OPT}")

km_final = KMeans(n_clusters=K_OPT, random_state=42, n_init=20)
labels_km = km_final.fit_predict(X_pca)
df_feat["cluster_km"] = labels_km

sil_km = silhouette_score(X_pca, labels_km, sample_size=5000, random_state=42)
print(f"Silhouette KMeans (k={K_OPT}): {sil_km:.4f}")
print("\nDistribución:")
print(df_feat["cluster_km"].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
palette = cm.tab10(np.linspace(0, 1, K_OPT))
for k in range(K_OPT):
    mask = labels_km == k
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1], s=5, alpha=0.4, color=palette[k], label=f"C{k}")
ax.set_title(f"UMAP — KMeans k={K_OPT}", fontsize=13)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## Gaussian Mixture Model (GMM)

The GMM models each cluster as a multivariate Gaussian, admits elliptical clusters and produces membership probabilities (usable in M2 to weight predictions by cluster). The number of components is selected by BIC.

In [ ]:
K_GMM_RANGE = range(3, 10)
bic_list, aic_list = [], []

for k in K_GMM_RANGE:
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42, max_iter=300)
    gmm.fit(X_pca)
    bic_list.append(gmm.bic(X_pca))
    aic_list.append(gmm.aic(X_pca))
    print(f"k={k}  BIC={bic_list[-1]:,.0f}  AIC={aic_list[-1]:,.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_GMM_RANGE, bic_list, marker="o", label="BIC", color="navy")
ax.plot(K_GMM_RANGE, aic_list, marker="s", label="AIC", color="darkorange")
ax.set_xlabel("k")
ax.set_title("GMM — BIC / AIC (↓ mejor)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

K_GMM = list(K_GMM_RANGE)[int(np.argmin(bic_list))]
print(f"\nk óptimo GMM (BIC): {K_GMM}")

In [ ]:
gmm_final = GaussianMixture(n_components=K_GMM, covariance_type="full", random_state=42, max_iter=300)
gmm_final.fit(X_pca)
labels_gmm  = gmm_final.predict(X_pca)
proba_gmm   = gmm_final.predict_proba(X_pca)  # membership probabilities (informative only; the final adopted model is KMeans)

df_feat["cluster_gmm"] = labels_gmm

sil_gmm = silhouette_score(X_pca, labels_gmm, sample_size=5000, random_state=42)
print(f"Silhouette GMM (k={K_GMM}): {sil_gmm:.4f}")
print(f"Confianza media de asignación: {proba_gmm.max(axis=1).mean():.3f}")
print("\nDistribución:")
print(pd.Series(labels_gmm).value_counts().sort_index())

## HDBSCAN

HDBSCAN does not require specifying k and detects outliers (label -1). In this pipeline it serves two purposes:

1. Structure analysis: it indicates whether the data present natural clusters of varying density.
2. Outlier detection: users with label=-1 are atypical cases that in M2 may be handled separately (with no segment-level propensity estimate).

In v1, a small `min_cluster_size` generated 350 micro-clusters with an inflated silhouette. Here the smallest `min_cluster_size` that keeps the outliers ≤ 20% is searched for automatically.

In [ ]:
# ── Automatic search of min_cluster_size ────────────────────────────────────
print("Búsqueda de min_cluster_size para outliers ≤ 20%:")
print(f"{'min_cs':>8}  {'n_clusters':>10}  {'outliers':>10}  {'pct_noise':>10}")

hdb_candidates = []
for mcs in [100, 200, 300, 500, 750, 1000, 1500, 2000]:
    hdb_tmp = HDBSCAN(min_cluster_size=mcs, min_samples=30, metric="euclidean")
    lbl_tmp = hdb_tmp.fit_predict(X_umap)
    nc = len(set(lbl_tmp)) - (1 if -1 in lbl_tmp else 0)
    pn = (lbl_tmp == -1).mean()
    print(f"{mcs:>8}  {nc:>10}  {(lbl_tmp==-1).sum():>10,}  {pn:>9.1%}")
    hdb_candidates.append({"mcs": mcs, "n_clusters": nc, "pct_noise": pn, "labels": lbl_tmp})

# Select the smallest mcs with pct_noise ≤ 0.20 and a reasonable n_clusters (2-20)
df_hdb_cand = pd.DataFrame([{k: v for k, v in d.items() if k != "labels"} for d in hdb_candidates])
valid_hdb = df_hdb_cand[(df_hdb_cand["pct_noise"] <= 0.20) & (df_hdb_cand["n_clusters"] >= 2) & (df_hdb_cand["n_clusters"] <= 20)]

if not valid_hdb.empty:
    best_idx = valid_hdb.index[0]
    MCS_OPT = int(df_hdb_cand.loc[best_idx, "mcs"])
    labels_hdb = hdb_candidates[best_idx]["labels"]
    print(f"\n→ min_cluster_size seleccionado: {MCS_OPT}")
else:
    print("\nNo se encontró configuración óptima. Usando fallback mcs=1000.")
    MCS_OPT = 1000
    best_cand = next(d for d in hdb_candidates if d["mcs"] == MCS_OPT)
    labels_hdb = best_cand["labels"]

In [ ]:
df_feat["cluster_hdb"] = labels_hdb

n_clusters_hdb = len(set(labels_hdb)) - (1 if -1 in labels_hdb else 0)
n_noise        = (labels_hdb == -1).sum()

print(f"HDBSCAN final — min_cluster_size={MCS_OPT}")
print(f"Clusters: {n_clusters_hdb}   Outliers: {n_noise:,} ({n_noise/len(labels_hdb):.1%})")
print("\nDistribución:")
print(pd.Series(labels_hdb).value_counts().sort_index())

# Silhouette only if there are valid clusters and ≤20 clusters (avoid an artificial silhouette)
mask_core = labels_hdb != -1
unique_core = np.unique(labels_hdb[mask_core])

if len(unique_core) >= 2 and n_clusters_hdb <= 20:
    sil_hdb = silhouette_score(X_umap[mask_core], labels_hdb[mask_core], sample_size=5000, random_state=42)
    print(f"\nSilhouette HDBSCAN (core points): {sil_hdb:.4f}")
else:
    sil_hdb = np.nan
    print("Silhouette no calculada (fuera de rango)")

In [ ]:
# ── Visual comparison of the three methods ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

configs = [
    (labels_km,  f"KMeans k={K_OPT}",           False),
    (labels_gmm, f"GMM k={K_GMM}",              False),
    (labels_hdb, f"HDBSCAN ({n_clusters_hdb}cl.)", True),
]

for ax, (lbl, title, has_noise) in zip(axes, configs):
    uniq = np.unique(lbl)
    pal  = cm.tab10(np.linspace(0, 1, len(uniq)))
    for i, k in enumerate(uniq):
        m = lbl == k
        c = "lightgray" if (k == -1 and has_noise) else pal[i]
        l = "Outliers" if k == -1 else f"C{k}"
        ax.scatter(X_umap[m, 0], X_umap[m, 1], s=4, alpha=0.35, color=c, label=l)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(markerscale=3, fontsize=8)

plt.suptitle("Comparación de métodos — espacio UMAP", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Final Model Selection

### Selection criteria

| Criterion | Description | Weight |
|---|---|---|
| Silhouette | Inter-cluster separation vs intra-cluster compactness | Primary |
| Balance | No cluster exceeds 60% of users | Filter |
| Interpretability | Clusters semantically assignable | Tie-breaker |

HDBSCAN enters the comparison only if it produces between 2 and 20 clusters and ≤20% outliers; its main purpose is outlier analysis, not operational segmentation.

In [ ]:
candidatos = {}

# KMeans: already validated by balance in the selection of k
candidatos[f"KMeans (k={K_OPT})"] = sil_km

# GMM: check balance
gmm_max_pct = pd.Series(labels_gmm).value_counts(normalize=True).max()
if gmm_max_pct <= 0.60:
    candidatos[f"GMM (k={K_GMM})"] = sil_gmm
else:
    print(f"GMM excluido: cluster dominante = {gmm_max_pct:.1%}")

# HDBSCAN: only if 2-20 clusters and <=20% outliers
hdb_ok = (2 <= n_clusters_hdb <= 20) and not np.isnan(sil_hdb) and (n_noise / len(labels_hdb) <= 0.20)
if hdb_ok:
    candidatos[f"HDBSCAN ({n_clusters_hdb} cl.)"] = sil_hdb
else:
    print(f"HDBSCAN excluido: {n_clusters_hdb} clusters, {n_noise/len(labels_hdb):.1%} outliers")

# Sort the candidates by silhouette, highest to lowest (a Series is simpler than sorted+lambda)
candidatos_ordenados = pd.Series(candidatos).sort_values(ascending=False)

print("\nCandidatos finales — Coeficiente de Silueta:")
for nombre, val in candidatos_ordenados.items():
    print(f"  {nombre:<30}  {val:.4f}")

# The selected model is the one with the highest silhouette (the first in the sorted list)
MODELO_SEL = candidatos_ordenados.index[0]
print(f"\n→ Modelo seleccionado: {MODELO_SEL}")

if "KMeans" in MODELO_SEL:
    df_feat["segment"] = df_feat["cluster_km"]
elif "GMM" in MODELO_SEL:
    df_feat["segment"] = df_feat["cluster_gmm"]
else:
    df_feat["segment"] = df_feat["cluster_hdb"]

print("\nDistribución del segmento final:")
print(df_feat["segment"].value_counts().sort_index())

## Segment Profiling

The mean features per segment are analysed in order to assign semantic labels, combining the behavioural variables with the demographic ones from `users.csv` (age, gender, employment status). The demographics did not take part in the clustering; their consistency within the clusters serves as external validation of the behavioural segmentation.

In [ ]:
df_seg_users = df_users.merge(
    df_feat[["id_user", "segment", "cluster_km", "cluster_gmm", "cluster_hdb",
             "n_eventos", "n_clicks", "n_opens", "click_rate",
             "n_productos", "n_sectores", "recencia_dias"]],
    on="id_user",
    how="left"
)

print(f"Usuarios con segmento: {df_seg_users['segment'].notna().sum():,} / {len(df_seg_users):,}")

# Mark man/employed as 1/0 to compute the % with a simple mean (without lambdas)
df_seg_users["es_hombre"]   = (df_seg_users["gender"] == "H").astype(int)
df_seg_users["es_empleado"] = (df_seg_users["labor_status"] == "employed").astype(int)

# Summary profile by segment
perfil = df_seg_users.groupby("segment").agg(
    n_usuarios    = ("id_user",       "count"),
    edad_media    = ("age",           "mean"),
    pct_hombre    = ("es_hombre",     "mean"),
    pct_empleado  = ("es_empleado",   "mean"),
    pct_coche     = ("tiene_coche",   "mean"),
    n_eventos_med = ("n_eventos",     "mean"),
    click_rate    = ("click_rate",    "mean"),
    recencia_med  = ("recencia_dias", "mean"),
)

# Convert pct_hombre and pct_empleado to a percentage (0-100)
perfil["pct_hombre"]   = perfil["pct_hombre"] * 100
perfil["pct_empleado"] = perfil["pct_empleado"] * 100
perfil = perfil.round(2)

perfil["pct_usuarios"] = (perfil["n_usuarios"] / len(df_seg_users) * 100).round(1)

display(perfil.sort_index())

In [ ]:
# ── Heatmap of normalised behavioural features ──────────────────────────────
PLOT_FEAT = ["n_eventos", "n_clicks", "n_opens", "click_rate", "recencia_dias",
             "n_productos", "n_sectores"]

seg_means = df_feat.groupby("segment")[PLOT_FEAT].mean()
seg_norm  = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min() + 1e-9)

fig, ax = plt.subplots(figsize=(max(8, len(seg_norm) * 0.9), 5))
sns.heatmap(
    seg_norm.T,
    annot=seg_means.T.round(2),
    fmt="g",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax
)
ax.set_title("Perfil comportamental por segmento (valores originales, colores normalizados)", fontsize=12)
ax.set_ylabel("Feature")
ax.set_xlabel("Segmento")
plt.tight_layout()
plt.show()

In [ ]:
# ── Sector and product distribution by segment ────────────────────────────
sec_cols  = [c for c in df_feat.columns if c.startswith("sec_")]
prod_cols = [c for c in df_feat.columns if c.startswith("prod_")]

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sec_by_seg = df_feat.groupby("segment")[sec_cols].mean()
sec_by_seg.columns = [c.replace("sec_", "") for c in sec_by_seg.columns]
sec_by_seg.T.plot(kind="bar", ax=axes[0], colormap="tab10")
axes[0].set_title("Frecuencia sectorial por segmento")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(title="Seg", fontsize=8, loc="upper right")

prod_by_seg = df_feat.groupby("segment")[prod_cols].mean()
prod_by_seg.columns = [c.replace("prod_", "") for c in prod_by_seg.columns]
prod_by_seg.T.plot(kind="bar", ax=axes[1], colormap="tab10")
axes[1].set_title("Preferencia de categoría de producto por segmento")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Seg", fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# ── Demographic distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
segs = sorted(df_feat["segment"].unique())

# Age
ax = axes[0, 0]
for s in segs:
    sub = df_seg_users[df_seg_users["segment"] == s]["age"].dropna()
    ax.hist(sub, bins=20, alpha=0.5, label=f"S{s}", density=True)
ax.set_title("Edad por segmento")
ax.legend(fontsize=7)

# Click rate
ax = axes[0, 1]
data = [df_seg_users[df_seg_users["segment"] == s]["click_rate"].dropna().values for s in segs]
ax.boxplot(data, labels=[f"S{s}" for s in segs])
ax.set_title("Click Rate por segmento")

# N events
ax = axes[0, 2]
data = [df_seg_users[df_seg_users["segment"] == s]["n_eventos"].dropna().values for s in segs]
ax.boxplot(data, labels=[f"S{s}" for s in segs])
ax.set_title("N° Eventos por segmento")

# Gender
ax = axes[1, 0]
gp = df_seg_users.groupby("segment")["gender"].value_counts(normalize=True).unstack().fillna(0) * 100
gp.plot(kind="bar", ax=ax, color=["steelblue", "salmon"])
ax.set_title("Género (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=8)

# Labor status
ax = axes[1, 1]
lp = df_seg_users.groupby("segment")["labor_status"].value_counts(normalize=True).unstack().fillna(0) * 100
lp.plot(kind="bar", ax=ax)
ax.set_title("Estado laboral (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=8)

# Recency
ax = axes[1, 2]
data = [df_seg_users[df_seg_users["segment"] == s]["recencia_dias"].dropna().values for s in segs]
ax.boxplot(data, labels=[f"S{s}" for s in segs])
ax.set_title("Recencia (días) por segmento")

plt.suptitle("Perfilado demográfico y comportamental por segmento", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary of Decisions

A compilation of the methodological decisions of the v2 segmentation.

In [ ]:
# ── Executive summary of decisions ─────────────────────────────────────────
sil_hdb_str = f"{sil_hdb:.4f}" if not np.isnan(sil_hdb) else "N/A"
n_seg       = df_feat["segment"].nunique()
seg_min     = df_feat["segment"].value_counts().min()
seg_max     = df_feat["segment"].value_counts().max()
n_sin_seg   = df_seg_users["segment"].isna().sum()

print("=" * 65)
print("RESUMEN DE DECISIONES — M1 Segmentación v2")
print("=" * 65)
print(f"""
FEATURES
  Bloques:  actividad ({len(ACT_COLS)}) + sector ({len(SEC_COLS)}) + producto ({len(PROD_COLS)})
  Total: {len(ALL_FEAT)} features  →  reducidas a {N_PCA} componentes PCA (≥90% varianza)
  log1p aplicado a: {LOG_COLS}
  Novedad v2: product_new (top-20 categorías) + click_rate por sector

CLUSTERING (sobre espacio PCA)
  KMeans  k={K_OPT:<2}  sil={sil_km:.4f}  (K_RANGE=4-12, filtro max_cluster≤60%)
  GMM     k={K_GMM:<2}  sil={sil_gmm:.4f}  (BIC óptimo, covariance=full)
  HDBSCAN k={n_clusters_hdb:<2}  sil={sil_hdb_str}  (mcs={MCS_OPT}, outliers={n_noise/len(labels_hdb):.1%})

MODELO SELECCIONADO: {MODELO_SEL}
  Criterio primario: mayor silhouette score
  Filtros aplicados: max_cluster≤60% | HDBSCAN solo si 2-20 clusters y ≤20% outliers

RESULTADO FINAL
  Segmentos: {n_seg}  |  Rango: {seg_min:,}–{seg_max:,} usuarios/segmento
  Usuarios sin segmento: {n_sin_seg} (sin historial de eventos)
""")

print("PERFIL POR SEGMENTO:")
print(f"  {'Segmento':<12}  {'N':>6}  {'%':>5}  {'click_rate':>10}  {'eventos':>7}  {'top_sector'}")
print("  " + "-" * 65)
for seg in sorted(df_feat["segment"].unique()):
    row   = perfil.loc[seg]
    sec   = df_feat[df_feat["segment"] == seg][sec_cols].mean()
    top_s = sec.idxmax().replace("sec_", "")
    label = "outliers" if seg == -1 else f"Seg {seg}"
    print(f"  {label:<12}  {int(row['n_usuarios']):>6,}  {row['pct_usuarios']:>4.1f}%"
          f"  {row['click_rate']:>10.3f}  {row['n_eventos_med']:>7.1f}  {top_s}")

## Export

`users_segmented.csv` is exported with:
- All attributes from `users.csv` (demographic + geographic)
- `segment`: cluster of the selected model (used in M2–M4)
- `cluster_km`, `cluster_gmm`, `cluster_hdb`: alternative assignments
- Behavioural features: `n_eventos`, `n_clicks`, `click_rate`, etc.
- UMAP coordinates for future visualisation

In [ ]:
df_feat["umap_x"] = X_umap[:, 0]
df_feat["umap_y"] = X_umap[:, 1]

df_out = df_seg_users.merge(
    df_feat[["id_user", "umap_x", "umap_y"]],
    on="id_user",
    how="left"
)

df_out.to_csv(PROCESSED_PATH / "users_segmented.csv", index=False, encoding="utf-8")

print(f"Exportado: users_segmented.csv  →  {df_out.shape}")
print(f"Segmentos: {df_out['segment'].nunique()}  |  Nulos: {df_out['segment'].isna().sum()}")
display(df_out[["id_user", "age", "gender", "labor_status",
                "segment", "n_eventos", "n_clicks", "click_rate",
                "umap_x", "umap_y"]].head(5))